This script is designed to process a CSV file containing information about BEAD ineligible locations. It reads the data, filters it based on specific criteria, and generates summary CSV files for different categories of ineligibility. The script uses the pandas library for data manipulation and analysis.

In [1]:
import pandas as pd
from os.path import join

Define the input and output variables. The BEAD data is downloaded from the MIHI website listed below.

https://www.michigan.gov/leo/bureaus-agencies/mihi/funding-opportunities/bead/bead-grant-application-website/bead-eligible-locations

In [2]:
text = pd.read_csv(r"C:\Users\mtcuster\Documents\ArcGIS\Projects\Local empowerment\rawdata08192026\Ineligible Locations.csv",
                   low_memory=False)
deposit_location = r'C:\Users\mtcuster\Documents\ArcGIS\Projects\Local empowerment\processeddata08192026'

The following two lines are intended to remove any rows that have a null value in the 'application_area' column and to drop the 'location_id' column from the DataFrame. This is done to clean the data and focus on relevant information for further analysis.

In [3]:
text = text.dropna(subset=['application_area'])
text = text.drop(columns=['location_id'])

now split the data into three separate DataFrames based on the 'post_challenge_eligibility' column. Each DataFrame will contain rows corresponding to a specific ineligibility reason: "Ineligible - Funded", "Ineligible - Planned Service", and "Ineligible - Served". This allows for targeted analysis of each category. Then count the number of Broadband serviceable locations (BSL) that are funded and create a new DataFrame that summarizes the count of BSLs for each application area. Additionally, a new column named 'fund_type' is added to the summary DataFrame, which will later be populated with the types of funding associated with each application area.

In [4]:
funded = text[text['post_challenge_eligibility'] == "Ineligible - Funded"]
planned_service = text[text['post_challenge_eligibility'] == "Ineligible - Planned Service"]
served = text[text['post_challenge_eligibility'] == "Ineligible - Served"]
simple_funded = funded.groupby(['application_area']).size().reset_index(name="sum of BSLs")
simple_funded.insert(2, "fund_type", "", True)

Groups the funded DataFrame by the application_area column, counts the number of rows in each group, and turns the result into a new flat DataFrame with columns application_area and sum of BSLs. Then exports to a csv file to the deposit locations.

In [5]:
def unique_values(old_df, name):
    new_df = old_df.groupby(['application_area']).size().reset_index(name="sum of BSLs")
    new_df.to_csv(join(deposit_location, name), index=False)

Groups the funded DataFrame by the application_area column, counts the number of rows in each group, and turns the result into a new flat DataFrame with columns application_area and sum of BSLs.

In [6]:
reduced_funded = funded.groupby(['application_area']).size().reset_index(name="sum of BSLs")

The out variable splits the fund_type column so that each funding program exists in its own column. Then the fund type column is added to the funded data frame. We remove any duplicates from the df_split based on the  application_area column.

In [7]:
out = funded['fund_type'].apply(lambda x: pd.Series(x.split(',')))
df_split = pd.concat([funded.drop('fund_type', axis=1), out], axis=1)
funded_areas = pd.unique(df_split['application_area'])

In [8]:
for alpha in funded_areas:
    selected_rows = df_split[df_split['application_area'] == alpha]
    concat_list = [*selected_rows.iloc[:, 4].unique(), *selected_rows.iloc[:, 5].unique(),
                   *selected_rows.iloc[:, 6].unique()]
    beta = pd.Series(concat_list).dropna().astype(str)
    gamma = beta.str.replace(" ", "", regex=False).unique()
    sigma = ", ".join(gamma)
    simple_funded.loc[simple_funded['application_area'] == alpha, "fund_type"] = sigma

creates ouputs

In [9]:
unique_values(served, "Ineligible_served.csv")
unique_values(planned_service, "Ineligible_planned_service.csv")
simple_funded.to_csv(join(deposit_location, "Ineligible_funded.csv"), index=False)

The next block is to deal with the No BEAD locations csv. The data is read from a CSV file, and any rows with null values in the 'application_area' column are dropped. The 'location_id', 'post_challenge_eligibility', 'fund_type', and 'Review_Status' columns are also removed from the DataFrame. Unique application areas are identified, and a filtered DataFrame is created based on these unique areas. A cross-tabulation table is generated to summarize the counts of reason codes for each application area, and the resulting table is exported to a CSV file.

In [10]:
text = pd.read_csv(r"C:\Users\mtcuster\Documents\ArcGIS\Projects\Local empowerment\rawdata08192026\No BEAD Locations.csv", low_memory=False)

text = text.dropna(subset=['application_area'])
text = text.drop(columns=['location_id', "post_challenge_eligibility", "fund_type", 'Review_Status'])
hexbins = pd.unique(text['application_area'])

# Filter the DataFrame for your set first
filtered_df = text[text['application_area'].isin(hexbins)]

# Create a cross-tabulation table
table = (pd.crosstab(filtered_df["application_area"], filtered_df["reason_code"]).add_prefix("Reason code ").rename_axis(columns=None).reset_index())

table.to_csv(join(deposit_location, "No_BEAD_Summerized.csv"), index=True)